# AI工学101 — 第5回
## NumPyで線形代数：`y = Wx + b` を自分の手で計算する

前回までで **配列 → ベクトル化 → データ抽出 → shape/axis** と進みました。

今日はとうとう、それらを一本につなげます。

\[
\boxed{y = Wx+b}
\]

ニューラルネットワークを開けると何度も出てくる、あいつです。👨‍🏫🌿

**所要時間：60〜90分**  
講義：約20分 ／ 実習：約40〜50分 ／ 演習：約20分

---

## 📖 講義：なぜAIで線形代数を使うのか

たとえば、ある人を

```text
年齢 = 30
身長 = 160
体重 = 55
```

と表現するとします。

NumPyなら、

```python
x = np.array([30, 160, 55])
```

ですね。

数学的には、

\[
x=
\begin{bmatrix}
30\\
160\\
55
\end{bmatrix}
\]

という**ベクトル**として扱えます。

つまり、

> 1人の人間 → 特徴量のベクトル

に変換したわけです。

画像だろうが文章だろうが音声だろうが、AIではこうやって数値表現に落とします。

---

## ① 内積

まず、

```python
x = np.array([1, 2, 3])
w = np.array([0.5, 1.0, -1.0])
```

として、

```python
print(np.dot(x, w))
```

を実行します。

内部では、

\[
1(0.5)+2(1.0)+3(-1.0)
\]

なので、

\[
0.5+2-3=-0.5
\]

となります。

つまり内積は、

> **各特徴量に重みを付けて全部足す**

操作だと見ることもできます。

これ、もう機械学習っぽいでしょう。

---

# 💻 実習1：内積を自力で確認する

```python
import numpy as np

x = np.array([2, 3, 4])
w = np.array([0.5, 2.0, -1.0])

y = np.dot(x, w)

print(y)
```

今度はNumPyを使わず、

```python
y_manual = (
    x[0] * w[0]
    + x[1] * w[1]
    + x[2] * w[2]
)

print(y_manual)
```

も実行してください。

同じ結果になるはずです。

ここで、

```python
np.dot(x, w)
```

が魔法ではなく、

```text
掛ける
 ↓
足す
```

をまとめてやっているだけだと確認できます。

---

# 📖 ② 行列積

今度は複数の重みを扱います。

```python
W = np.array([
    [1, 2],
    [3, 4]
])

x = np.array([
    [10],
    [20]
])
```

shapeを確認。

```python
print(W.shape)
print(x.shape)
```

結果：

```text
(2, 2)
(2, 1)
```

そして、

```python
y = W @ x

print(y)
```

計算は、

\[
\begin{bmatrix}
1&2\\
3&4
\end{bmatrix}
\begin{bmatrix}
10\\
20
\end{bmatrix}
\]

なので、

\[
\begin{bmatrix}
1(10)+2(20)\\
3(10)+4(20)
\end{bmatrix}
=
\begin{bmatrix}
50\\
110
\end{bmatrix}
\]

になります。

---

## shapeだけで計算可能か判断できる

ここ重要。

```text
W     @    x

(2,2)      (2,1)
   ↑        ↑
   └── 2 = 2 ──┘
```

真ん中の数字が一致しているので計算できます。

結果のshapeは**外側**が残って、

```text
(2,1)
```

です。

一般化すると、

```text
(m, n) @ (n, p)

       ↓

     (m, p)
```

これを覚えるとshapeエラーへの耐性が猛烈に上がります。

---

# 💻 実習2：shapeを予想する

実行する**前に**答えてください。

```python
A = np.ones((3, 4))
B = np.ones((4, 2))

C = A @ B
```

`C.shape` は？

考え方は、

```text
(3, 4) @ (4, 2)
```

だから……

実際に確認。

```python
print(C.shape)
```

---

# 📖 ③ 転置 `T`

次はこちら。

```python
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print(A.shape)
```

```text
(2, 3)
```

転置すると、

```python
print(A.T)
```

```text
[[1 4]
 [2 5]
 [3 6]]
```

shapeは、

```text
(3, 2)
```

になります。

つまり、

\[
A_{2\times3}
\rightarrow
A^T_{3\times2}
\]

です。

機械学習では転置も頻出します。

---

# 💻 実習3：いよいよ `Wx+b`

入力：

```python
x = np.array([
    [2],
    [3]
])
```

重み：

```python
W = np.array([
    [0.5, 1.0],
    [-1.0, 2.0]
])
```

バイアス：

```python
b = np.array([
    [0.1],
    [0.2]
])
```

計算します。

```python
y = W @ x + b

print(y)
```

はい。

これです。

\[
\boxed{y=Wx+b}
\]

---

## 何が起きている？

まず、

```python
W @ x
```

で入力に重みを掛ける。

その後、

```python
+ b
```

でバイアスを加える。

つまり、

```text
入力 x
 │
 ▼
重み W
 │
 ▼
W @ x
 │
 ▼
バイアス +b
 │
 ▼
出力 y
```

となります。

この計算を大量に組み合わせたものがニューラルネットワークの基本構造です。

---

# 💻 実習4：複数サンプルを一気に処理

ここから今日の本丸。

3人のデータがあります。

```python
X = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [4.0, 1.0]
])
```

shape：

```text
(3, 2)
```

つまり、

```text
3サンプル × 2特徴量
```

です。

重みを、

```python
W = np.array([
    [0.5],
    [2.0]
])
```

とします。

shape：

```text
(2,1)
```

すると、

```python
y = X @ W

print(y)
```

が計算できます。

shapeを追うと、

```text
(3,2) @ (2,1)

       ↓

     (3,1)
```

つまり、

> **3人分をfor文なしで一発計算**

しています。

第2回の「ベクトル化」がここにつながった。

---

# 💻 実習5：バイアスも追加

```python
b = 0.5

y = X @ W + b

print(y)
```

第2回でやったブロードキャストによって、すべてのサンプルに `0.5` が加算されます。

ここまでの知識が、

```text
ndarray
  ↓
shape
  ↓
ベクトル化
  ↓
broadcast
  ↓
行列積
  ↓
Wx+b
```

と一本につながりました。

---

# ✍️ 演習：小さな人工ニューロンを作る

次の入力を使います。

```python
X = np.array([
    [1.0, 3.0],
    [2.0, 1.0],
    [4.0, 2.0],
    [5.0, 3.0]
])
```

これは、

```text
4サンプル × 2特徴量
```

です。

重みは、

```python
W = np.array([
    [0.7],
    [-0.4]
])
```

バイアスは、

```python
b = 0.2
```

とします。

今回の課題は4つ。

1. `X.shape` と `W.shape` を確認する。
2. **コードを実行する前に** `X @ W` のshapeを予想する。
3. `y = X @ W + b` を計算する。
4. 1サンプル目について、NumPyを使わず手計算し、結果が一致するか確認する。

最後は、

\[
1.0(0.7)+3.0(-0.4)+0.2
\]

を計算すればOK。

---

## 🌿 第5回の核心

今日覚えてほしいのは関数名より、この構造です。

\[
\boxed{y=Wx+b}
\]

入力 `x` に対して、学習可能なパラメータ `W` と `b` を使い、別の数値表現 `y` へ変換する。

まだ**学習は一切していません**。

今は僕らが `W` を手入力しています。

ではAIの「学習」とは何なのか？

ものすごく乱暴に先取りすると、

> **正解に近づくように、この `W` と `b` を自動調整すること**

です。

ここ、重要。

ニューラルネットワークという巨大な概念が、急に、

```python
y = X @ W + b
```

という君がもう読めるNumPyコードまで降りてきました。

次の**第6回は「統計量とデータ前処理」**。平均・分散・標準偏差・標準化をNumPyで実装して、いよいよ「配列操作」から**機械学習用データを作る工程**へ進みます。👨‍🏫🥝

In [10]:
# # ✍️ 演習：小さな人工ニューロンを作る

import numpy as np

# 次の入力を使います。

# ```python
X = np.array([
    [1.0, 3.0],
    [2.0, 1.0],
    [4.0, 2.0],
    [5.0, 3.0]
])

# 4サンプル × 2特徴量
# ```

# 重みは、

# ```python
W = np.array([
    [0.7],
    [-0.4]
])
# ```

# バイアスは、
# ```python
b = 0.2
# ```
# とします。

# 1. `X.shape` と `W.shape` を確認する。
print(X.shape)
print(W.shape)

# 2. **コードを実行する前に** `X @ W` のshapeを予想する。
print((X @ W).shape)

# 3. `y = X @ W + b` を計算する。
y = X @ W + b
print(y)

# 4. 1サンプル目について、NumPyを使わず手計算し、結果が一致するか確認する。
y_manual = (
    X[0, 0] * W[0]
    +X[0, 1] * W[1]
)
print(y_manual + b)

(4, 2)
(2, 1)
(4, 1)
[[-0.3]
 [ 1.2]
 [ 2.2]
 [ 2.5]]
[-0.3]
